# 04 — Baseline Models: SVM & KNN

Baseline classifiers for comparison with CNN-LSTM on the same T&FD feature set.
These correspond to the benchmark table in the published paper.

## 1. Load Data

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_curve, roc_auc_score,
                             f1_score)
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings("ignore")

data = pd.read_csv("data/processed/Training_Data.csv")
X = data.drop(data.columns[0], axis=1)
y = data[data.columns[0]]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_res, y_res = SMOTE(random_state=42).fit_resample(X, y_encoded)
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42, stratify=y_res
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Classes: {label_encoder.classes_}")

## 2. SVM

In [ ]:
from sklearn.svm import SVC

svm_model = SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=42)
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)

print("── SVM Results ──")
print(f"Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}")
print(f"F1 macro: {f1_score(y_test, y_pred_svm, average='macro'):.4f}")
print(classification_report(y_test, y_pred_svm, target_names=label_encoder.classes_))

In [ ]:
plt.figure(figsize=(5,4))
sns.heatmap(confusion_matrix(y_test, y_pred_svm), annot=True, fmt="d",
            cmap="Blues", xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title("Confusion Matrix — SVM", fontsize=11, fontweight="bold")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout()
plt.savefig("results/figures/confusion_matrix_svm.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Grid search for best K
best_k, best_acc = 3, 0
for k in [3, 5, 7, 9, 11]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    acc = accuracy_score(y_test, knn.predict(X_test))
    print(f"  K={k}: {acc:.4f}")
    if acc > best_acc:
        best_acc, best_k = acc, k

knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_test)

print(f"\nBest K={best_k}")
print(classification_report(y_test, y_pred_knn, target_names=label_encoder.classes_))

## 4. Comparison Table

In [ ]:
print(f"{'Model':<12} {'Accuracy':>10} {'F1 (macro)':>12}")
print("-" * 36)
for name, preds in [("SVM", y_pred_svm), ("KNN", y_pred_knn)]:
    print(f"{name:<12} {accuracy_score(y_test, preds):>10.4f} "
          f"{f1_score(y_test, preds, average='macro'):>12.4f}")
print()
print("Note: CNN-LSTM-SW achieves 97% train-position detection. See notebook 03.")